# EXPLAIN ANALYZE — Diagnóstico de Queries

Referência de diagnóstico de performance no PostgreSQL. Schema usado: fintech de crédito pessoal, ~2M propostas.

## Como o PostgreSQL Executa uma Query

![Fluxo de execução de queries no PostgreSQL](assets/01-query-execution-flow.png)

**Parser/Rewriter**: valida sintaxe e expande views. Raramente relevante para diagnóstico.

**Planner/Optimizer**: o que importa. Lê `pg_statistic` para estimar seletividade e custo de cada plano alternativo. Quando ele erra a estimativa (ex: espera 41 linhas e retorna 39.000), é aqui que o problema começa.

**Executor**: roda o plano escolhido, nó por nó. O `ANALYZE` do EXPLAIN mede o tempo real nessa etapa.

## Schema Base — Sistema de Propostas de Crédito

In [ ]:
# Instalação das dependências (execute uma vez)
# !pip install psycopg2-binary sqlalchemy pandas

import psycopg2
import psycopg2.extras
import json
import re
from textwrap import dedent

# Configuração da conexão
# Ajuste as credenciais para seu ambiente local
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "fintech_kb",
    "user": "postgres",
    "password": "postgres",
}


def get_connection():
    """Retorna uma conexão com o banco de dados."""
    return psycopg2.connect(**DB_CONFIG)


def run_sql(sql, params=None, fetch=True):
    """Executa um SQL e retorna os resultados como lista de dicionários."""
    with get_connection() as conn:
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(sql, params)
            if fetch:
                return [dict(row) for row in cur.fetchall()]
            conn.commit()
            return None


def explain(sql, analyze=False, buffers=False, fmt="text"):
    """
    Executa EXPLAIN sobre um SQL e imprime o plano formatado.

    analyze: se True, executa a query de verdade e mede tempo real
    buffers: se True, inclui estatísticas de buffer (apenas com analyze=True)
    fmt: 'text' ou 'json'
    """
    options = []
    if analyze:
        options.append("ANALYZE")
    if buffers and analyze:
        options.append("BUFFERS")
    options.append(f"FORMAT {fmt.upper()}")

    explain_sql = f"EXPLAIN ({', '.join(options)}) {sql}"

    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(explain_sql)
            rows = cur.fetchall()

    if fmt == "text":
        print("\n".join(row[0] for row in rows))
    else:
        # FORMAT JSON retorna uma única linha com o JSON completo
        plan = rows[0][0]
        print(json.dumps(plan, indent=2))


print("Funções auxiliares carregadas.")

In [ ]:
# Criação do schema base
# Execute este bloco uma única vez para preparar o ambiente de estudo

CREATE_SCHEMA = """
DROP TABLE IF EXISTS parcelas CASCADE;
DROP TABLE IF EXISTS decisoes_log CASCADE;
DROP TABLE IF EXISTS consultas_externas CASCADE;
DROP TABLE IF EXISTS propostas CASCADE;
DROP TABLE IF EXISTS clientes CASCADE;

CREATE TABLE clientes (
    id          SERIAL PRIMARY KEY,
    cpf         VARCHAR(11) UNIQUE NOT NULL,
    nome        VARCHAR(200) NOT NULL,
    renda_mensal NUMERIC(12,2),
    created_at  TIMESTAMP DEFAULT NOW()
);

CREATE TABLE propostas (
    id         SERIAL PRIMARY KEY,
    numero     VARCHAR(20) UNIQUE NOT NULL,
    cliente_id INTEGER REFERENCES clientes(id),
    valor      NUMERIC(12,2) NOT NULL,
    status     VARCHAR(20) NOT NULL DEFAULT 'pendente',
    score      INTEGER,
    created_at TIMESTAMP DEFAULT NOW()
);

CREATE TABLE consultas_externas (
    id           SERIAL PRIMARY KEY,
    proposta_id  INTEGER REFERENCES propostas(id),
    provider     VARCHAR(50) NOT NULL,
    response_data JSONB,
    status       VARCHAR(20) NOT NULL,
    created_at   TIMESTAMP DEFAULT NOW()
);

CREATE TABLE decisoes_log (
    id              SERIAL PRIMARY KEY,
    proposta_id     INTEGER REFERENCES propostas(id),
    etapa           VARCHAR(50) NOT NULL,
    resultado       VARCHAR(20) NOT NULL,
    regras_aplicadas JSONB,
    executed_at     TIMESTAMP DEFAULT NOW()
);

CREATE TABLE parcelas (
    id              SERIAL PRIMARY KEY,
    proposta_id     INTEGER REFERENCES propostas(id),
    numero          INTEGER NOT NULL,
    valor           NUMERIC(12,2) NOT NULL,
    vencimento      DATE NOT NULL,
    status_pagamento VARCHAR(20) DEFAULT 'pendente',
    pago_em         TIMESTAMP
);
"""

run_sql(CREATE_SCHEMA, fetch=False)
print("Schema criado com sucesso.")

In [ ]:
# Geração de dados de teste
# ~200.000 clientes, ~2.000.000 propostas, ~2.000.000 parcelas
# Volume necessário para tornar os problemas de performance visíveis

POPULATE = """
-- Clientes
INSERT INTO clientes (cpf, nome, renda_mensal, created_at)
SELECT
    LPAD(i::text, 11, '0'),
    'Cliente ' || i,
    (random() * 15000 + 1500)::numeric(12,2),
    NOW() - (random() * INTERVAL '2 years')
FROM generate_series(1, 200000) AS s(i);

-- Propostas (10 por cliente em media)
INSERT INTO propostas (numero, cliente_id, valor, status, score, created_at)
SELECT
    'PROP-' || LPAD(i::text, 10, '0'),
    (random() * 199999 + 1)::int,
    (random() * 50000 + 1000)::numeric(12,2),
    CASE (random() * 4)::int
        WHEN 0 THEN 'pendente'
        WHEN 1 THEN 'aprovada'
        WHEN 2 THEN 'reprovada'
        ELSE 'cancelada'
    END,
    (random() * 1000)::int,
    NOW() - (random() * INTERVAL '2 years')
FROM generate_series(1, 2000000) AS s(i);

-- Parcelas para propostas aprovadas (12 parcelas cada)
-- Vencimentos de -10 meses a +2 meses para ter parcelas vencidas
INSERT INTO parcelas (proposta_id, numero, valor, vencimento, status_pagamento, pago_em)
SELECT
    p.id,
    n.numero,
    (p.valor / 12)::numeric(12,2),
    (NOW() - INTERVAL '10 months' + (n.numero * INTERVAL '1 month'))::date,
    CASE WHEN n.numero <= 6 THEN 'pago' ELSE 'pendente' END,
    CASE WHEN n.numero <= 6
         THEN NOW() - (random() * INTERVAL '180 days')
         ELSE NULL
    END
FROM propostas p
CROSS JOIN generate_series(1, 12) AS n(numero)
WHERE p.status = 'aprovada'
LIMIT 2000000;

-- Decisoes log (3 etapas por proposta dos ultimos 60 dias)
INSERT INTO decisoes_log (proposta_id, etapa, resultado, regras_aplicadas, executed_at)
SELECT
    p.id, etapa,
    CASE WHEN random() > 0.3 THEN 'aprovado' ELSE 'reprovado' END,
    '{"regra": "score_minimo"}'::jsonb,
    p.created_at + INTERVAL '1 hour'
FROM propostas p
CROSS JOIN (VALUES ('analise_credito'), ('consulta_bureau'), ('decisao_final')) AS e(etapa)
WHERE p.created_at >= NOW() - INTERVAL '60 days';

-- Atualiza estatísticas para o planner trabalhar com dados reais
ANALYZE clientes, propostas, parcelas, decisoes_log;
"""

run_sql(POPULATE, fetch=False)
print("Dados inseridos e estatísticas atualizadas.")

# Verificação dos volumes
counts = run_sql("""
    SELECT
        'clientes'  AS tabela, COUNT(*) AS total FROM clientes
    UNION ALL
    SELECT 'propostas', COUNT(*) FROM propostas
    UNION ALL
    SELECT 'parcelas',  COUNT(*) FROM parcelas
    UNION ALL
    SELECT 'decisoes_log', COUNT(*) FROM decisoes_log
""")
for row in counts:
    print(f"  {row['tabela']:15s}: {row['total']:>10,}")

## 1. Como Ler o Output do EXPLAIN ANALYZE

O plano de execução é uma árvore de nós. O PostgreSQL executa de dentro para fora (nós folha até a raiz).

### Anatomia de um nó

```
Seq Scan on propostas  (cost=0.00..48200.00 rows=2000000 width=68)
                        (actual time=0.012..380.4 rows=2000000 loops=1)
```

| Campo | Significado |
|---|---|
| `Seq Scan` | Tipo do nó (operação) |
| `cost=0.00..48200.00` | Custo estimado: startup..total (unidades abstratas, não ms) |
| `rows=2000000` | Linhas estimadas pelo planner |
| `width=68` | Bytes por linha estimado |
| `actual time=0.012..380.4` | Tempo real em ms: primeira linha..última linha |
| `rows=2000000` | Linhas reais retornadas |
| `loops=1` | Quantas vezes esse nó executou (em joins, loops > 1 é sinal de alerta) |

### Tipos de Scan

| Tipo | Quando aparece |
|---|---|
| `Seq Scan` | Sem índice útil ou retornando % grande da tabela. Normal em tabelas < 1000 linhas |
| `Index Scan` | Índice existe e poucas linhas passam no filtro. Acessa índice + heap |
| `Index Only Scan` | Tudo que a query precisa está no índice — não toca no heap |
| `Bitmap Heap Scan` | Seletividade média. Primeiro varre o índice, depois acessa o heap em batch |

### Tipos de Join

| Tipo | Quando o planner escolhe |
|---|---|
| `Nested Loop` | Uma tabela pequena, outra com índice bom. Se `loops` for alto sem índice, é problema |
| `Hash Join` | Tabelas médias/grandes. Constrói hash da menor, faz probe com a maior |
| `Merge Join` | Ambas já ordenadas (por índice ou Sort prévio) |

### Estimativa vs Real

Se `rows` estimado diverge muito do real (10x+), o planner tomou decisão ruim baseada em estatísticas velhas.
`ANALYZE tabela;` atualiza as estatísticas. Em prod, o autovacuum faz isso automaticamente, mas tabelas com muitos inserts podem defasar.

In [4]:
# Exemplo 1: Seq Scan — leitura completa da tabela
# Sem WHERE, o PostgreSQL precisa ler todas as 2.000.000 linhas

SQL_SEQ_SCAN = """
SELECT id, numero, status, valor
FROM propostas
"""

print("=== Seq Scan (sem filtro) ===")
explain(SQL_SEQ_SCAN, analyze=True)

=== Seq Scan (sem filtro) ===
Seq Scan on propostas  (cost=0.00..40619.00 rows=2000000 width=37) (actual time=0.168..675.526 rows=2000000 loops=1)
Planning Time: 5.031 ms
Execution Time: 779.088 ms


In [ ]:
# Exemplo 2: Index Scan — busca seletiva por chave primária
# A PK é sempre indexada. Retorna 1 linha: muito mais rápido.

SQL_INDEX_SCAN = """
SELECT id, numero, status, valor
FROM propostas
WHERE id = 12345
"""

print("=== Index Scan (filtro por PK) ===")
explain(SQL_INDEX_SCAN, analyze=True)

In [ ]:
# Exemplo 3: Index Only Scan — sem acessar a tabela
# Quando todos os campos da SELECT estão no índice, o PostgreSQL
# não precisa ir até o heap (a tabela em si).

# O campo 'numero' tem índice UNIQUE (criado automaticamente)
# Se selecionarmos apenas 'numero', o planner pode usar Index Only Scan

SQL_INDEX_ONLY = """
SELECT numero
FROM propostas
WHERE numero = 'PROP-0000012345'
"""

print("=== Index Only Scan (todos campos no índice) ===")
explain(SQL_INDEX_ONLY, analyze=True)

In [ ]:
# Exemplo 4: Hash Join — join entre tabelas sem ordenação prévia
# Típico para tabelas grandes. O PostgreSQL constrói uma hash table
# da tabela menor e faz probe com a maior.

SQL_HASH_JOIN = """
SELECT
    c.nome,
    p.numero,
    p.valor,
    p.status
FROM propostas p
JOIN clientes c ON c.id = p.cliente_id
WHERE p.status = 'aprovada'
LIMIT 100
"""

print("=== Hash Join (propostas x clientes) ===")
explain(SQL_HASH_JOIN, analyze=True)

## 2. Opções do EXPLAIN

| Comando | O que faz | Executa? |
|---|---|---|
| `EXPLAIN sql` | Plano estimado | Não |
| `EXPLAIN ANALYZE sql` | Plano + tempo real medido | Sim |
| `EXPLAIN (ANALYZE, BUFFERS) sql` | + estatísticas de cache vs disco | Sim |
| `EXPLAIN (ANALYZE, FORMAT JSON) sql` | Mesmo plano em JSON (para jogar no explain.dalibo.com) | Sim |

CUIDADO: `EXPLAIN ANALYZE` roda a query. Em DELETE/UPDATE, envolva em transação:

```sql
BEGIN;
EXPLAIN ANALYZE DELETE FROM propostas WHERE status = 'cancelada';
ROLLBACK;
```

O rollback desfaz a operação mas o tempo medido é real.

### Entendendo Buffers

```
Buffers: shared hit=1823 read=412
```

| Campo | O que significa na prática |
|---|---|
| `shared hit` | Páginas já estavam no buffer pool (RAM) |
| `shared read` | Páginas lidas do disco — aqui é onde dói |
| `shared written` | Páginas sujas escritas de volta |

Hit ratio = `hit / (hit + read)`. Abaixo de 95% em queries frequentes indica que `shared_buffers` pode estar pequeno ou que falta índice (scan lendo páginas demais).

In [ ]:
# Variante 1: EXPLAIN básico (sem executar)
# Útil para verificar o plano antes de rodar em produção

SQL = """
SELECT p.numero, p.valor, c.nome
FROM propostas p
JOIN clientes c ON c.id = p.cliente_id
WHERE p.created_at >= NOW() - INTERVAL '30 days'
  AND p.status = 'pendente'
"""

print("=== EXPLAIN (plano estimado, sem executar) ===")
explain(SQL)

In [9]:
# Variante 2: EXPLAIN ANALYZE (executa e mede)
# Compara o que o planner estimou com o que aconteceu de verdade

print("=== EXPLAIN ANALYZE (plano real + tempo) ===")
explain(SQL, analyze=True)

=== EXPLAIN ANALYZE (plano real + tempo) ===
Gather  (cost=38340.92..43503.70 rows=10609 width=38) (actual time=889.641..961.894 rows=11100 loops=1)
  Workers Planned: 1
  Workers Launched: 1
  ->  Parallel Hash Join  (cost=37340.92..41442.80 rows=6241 width=38) (actual time=873.498..902.779 rows=5550 loops=2)
        Hash Cond: (c.id = p.cliente_id)
        ->  Parallel Seq Scan on clientes c  (cost=0.00..3046.47 rows=117647 width=18) (actual time=0.009..11.598 rows=100000 loops=2)
        ->  Parallel Hash  (cost=37285.67..37285.67 rows=4420 width=28) (actual time=870.798..870.800 rows=5550 loops=2)
              Buckets: 16384  Batches: 1  Memory Usage: 864kB
              ->  Parallel Seq Scan on propostas p  (cost=0.00..37285.67 rows=4420 width=28) (actual time=0.640..865.017 rows=5550 loops=2)
                    Filter: (((status)::text = 'pendente'::text) AND (created_at >= (now() - '30 days'::interval)))
                    Rows Removed by Filter: 994450
Planning Time: 5.802 m

In [ ]:
# Variante 3: EXPLAIN (ANALYZE, BUFFERS)
# Mostra quantas páginas vieram da RAM vs do disco
# Execute duas vezes: a segunda deve ter mais 'hit' (cache quente)

print("=== EXPLAIN (ANALYZE, BUFFERS) ===")
print("--- Primeira execução (cache frio) ---")
explain(SQL, analyze=True, buffers=True)

print("\n--- Segunda execução (cache quente) ---")
explain(SQL, analyze=True, buffers=True)

In [ ]:
# Variante 4: FORMAT JSON
# Estrutura de dados completa do plano — útil para ferramentas
# como explain.dalibo.com ou pgMustard

print("=== EXPLAIN (FORMAT JSON) — primeiros campos do nó raiz ===")

with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute(f"EXPLAIN (ANALYZE, FORMAT JSON) {SQL}")
        plan_json = cur.fetchone()[0]

# plan_json já é uma lista Python (psycopg2 faz o parse do JSON)
root_plan = plan_json[0]["Plan"]

print("Tipo do nó raiz   :", root_plan.get("Node Type"))
print("Startup Cost      :", root_plan.get("Startup Cost"))
print("Total Cost        :", root_plan.get("Total Cost"))
print("Rows estimadas    :", root_plan.get("Plan Rows"))
print("Rows reais        :", root_plan.get("Actual Rows"))
print("Tempo real (ms)   :", root_plan.get("Actual Total Time"))
print("Planning Time(ms) :", plan_json[0].get("Planning Time"))
print("Execution Time    :", plan_json[0].get("Execution Time"))

## 3. pg_stat_statements — Identificando Queries Lentas

Extensão que acumula estatísticas de execução de todas as queries. Em prod, é onde você olha primeiro quando alguém reclama que "o sistema está lento".

### Habilitando

1. Em `postgresql.conf`:
   ```
   shared_preload_libraries = 'pg_stat_statements'
   pg_stat_statements.track = all
   ```
2. Reinicie o PostgreSQL
3. `CREATE EXTENSION pg_stat_statements;`

### Colunas que importam na prática

As que você vai usar mais:
- `mean_exec_time` + `calls`: queries frequentes e lentas. `total = mean * calls` mostra impacto real no sistema
- `shared_blks_read` alto com `shared_blks_hit` baixo: query batendo no disco, candidata a índice
- `query`: texto normalizado com `$1`, `$2`... Use `pg_stat_statements_reset()` para limpar após tuning

Menos urgentes: `rows`, `stddev_exec_time` (detecta queries com comportamento inconsistente — às vezes rápida, às vezes lenta)

In [ ]:
# Verifica se pg_stat_statements está disponível

CHECK_EXTENSION = """
SELECT name, default_version, installed_version
FROM pg_available_extensions
WHERE name = 'pg_stat_statements'
"""

result = run_sql(CHECK_EXTENSION)
if result:
    row = result[0]
    print(f"Extensão: {row['name']}")
    print(f"Versão disponível : {row['default_version']}")
    print(f"Versão instalada  : {row['installed_version'] or 'NÃO INSTALADA'}")
else:
    print("Extensão não encontrada.")

In [ ]:
# Instala a extensão (necessário uma vez por banco)
# Se falhar, veja as instruções acima sobre shared_preload_libraries

try:
    run_sql("CREATE EXTENSION IF NOT EXISTS pg_stat_statements;", fetch=False)
    print("Extensão pg_stat_statements instalada.")
except Exception as e:
    print(f"Não foi possível instalar: {e}")
    print("Adicione 'pg_stat_statements' em shared_preload_libraries e reinicie.")

In [ ]:
# Gera carga para popular pg_stat_statements com exemplos reais
# Simula queries típicas de um sistema de propostas

WORKLOAD_QUERIES = [
    # Query rápida: busca por PK
    "SELECT id, numero, status FROM propostas WHERE id = %s",
    # Query média: filtro por status
    "SELECT id, numero, valor FROM propostas WHERE status = %s",
    # Query lenta: filtro em coluna sem índice
    "SELECT id, numero, valor FROM propostas WHERE created_at >= NOW() - INTERVAL '30 days' AND status = %s",
    # Join sem índice
    "SELECT c.nome, COUNT(p.id) FROM clientes c JOIN propostas p ON p.cliente_id = c.id GROUP BY c.nome ORDER BY 2 DESC LIMIT 20",
]

import random

with get_connection() as conn:
    with conn.cursor() as cur:
        # 100 execuções de cada query
        for _ in range(100):
            cur.execute(WORKLOAD_QUERIES[0], (random.randint(1, 2000000),))
            cur.execute(WORKLOAD_QUERIES[1], ('pendente',))
            cur.execute(WORKLOAD_QUERIES[2], ('aprovada',))
        cur.execute(WORKLOAD_QUERIES[3])

print("Carga de trabalho gerada.")

In [ ]:
# Top 10 queries mais custosas do banco
# Ordenado por tempo total (total_exec_time): identifica o maior impacto

TOP_SLOW_QUERIES = """
SELECT
    calls,
    ROUND(total_exec_time::numeric, 2)   AS total_ms,
    ROUND(mean_exec_time::numeric, 2)    AS mean_ms,
    ROUND(stddev_exec_time::numeric, 2)  AS stddev_ms,
    rows,
    ROUND(
        100.0 * shared_blks_hit
        / NULLIF(shared_blks_hit + shared_blks_read, 0), 1
    ) AS cache_hit_pct,
    LEFT(query, 80) AS query_preview
FROM pg_stat_statements
WHERE query NOT ILIKE '%pg_stat%'
ORDER BY total_exec_time DESC
LIMIT 10
"""

try:
    results = run_sql(TOP_SLOW_QUERIES)
    print(f"{'CALLS':>6} {'TOTAL_MS':>10} {'MEAN_MS':>8} {'CACHE%':>7}  QUERY")
    print("-" * 80)
    for r in results:
        print(
            f"{r['calls']:>6} "
            f"{r['total_ms']:>10} "
            f"{r['mean_ms']:>8} "
            f"{str(r['cache_hit_pct'] or 'N/A'):>7}  "
            f"{r['query_preview']}"
        )
except Exception as e:
    print(f"Erro: {e}")
    print("Verifique se pg_stat_statements foi habilitado em shared_preload_libraries.")

## 4. Caso Real — Relatório de Propostas por Período Demorando 4s

### Contexto

O time de negócios reclama que o relatório de propostas dos últimos 30 dias está demorando mais de 4 segundos. Roda a cada 15 minutos via dashboard — ou seja, é uma query frequente batendo numa tabela de 2M de linhas.

Diagnóstico: EXPLAIN ANALYZE na query, achar o nó caro, decidir se índice resolve ou se é problema de schema/query.

In [ ]:
# Passo 1: A query do relatório (versão lenta)
# O time usa esta query para buscar propostas dos últimos 30 dias

RELATORIO_LENTO = """
SELECT
    p.numero,
    p.valor,
    p.status,
    p.score,
    p.created_at,
    c.nome          AS cliente_nome,
    c.cpf           AS cliente_cpf,
    c.renda_mensal
FROM propostas p
JOIN clientes c ON c.id = p.cliente_id
WHERE p.created_at >= NOW() - INTERVAL '30 days'
  AND p.status IN ('pendente', 'aprovada')
ORDER BY p.created_at DESC
"""

print("=== DIAGNÓSTICO: Relatório lento ===")
print()
explain(RELATORIO_LENTO, analyze=True, buffers=True)

In [ ]:
# Passo 2: Medir o tempo real de execução

import time

start = time.perf_counter()
result = run_sql(RELATORIO_LENTO)
elapsed = time.perf_counter() - start

print(f"Linhas retornadas : {len(result):,}")
print(f"Tempo de execução : {elapsed:.3f}s")

### Análise do Plano

Parallel Seq Scan em 2M linhas + Sort em disco (`external merge`). A query retorna ~325k linhas (14% da tabela) — é um volume grande demais para um índice ajudar significativamente.

### Índice Composto: `(created_at DESC, status)`

Índice criado, mas o planner continua usando Bitmap Heap Scan + Sort em disco. O speedup é ~0.9x — ou seja, sem melhora real.

Por que o índice não ajuda aqui?
- A query retorna 14% das linhas da tabela. Para volumes acima de ~5-10%, o custo de random I/O do índice supera o custo de Seq Scan sequencial
- O planner sabe disso e prefere Seq Scan mesmo com índice disponível
- O Sort em disco (`external merge`) acontece porque 325k linhas não cabem no `work_mem` (4MB default)

### Quando o índice ajuda de verdade

Compare com o Exercício 5 mais abaixo: `WHERE status = 'aprovada' ORDER BY created_at DESC LIMIT 50`. O índice `(status, created_at DESC)` reduz de 84ms para 0.3ms (265x mais rápido) porque:
- A query precisa de apenas 50 linhas (alta seletividade efetiva via LIMIT)
- O índice entrega as linhas já na ordem certa — sem Sort
- O planner para de ler após 50 entradas

A lição: **índice resolve filtros seletivos (< 5% das linhas) ou queries com ORDER BY + LIMIT. Para relatórios que retornam milhares de linhas, otimize o work_mem ou use materialized views.**

In [ ]:
# Passo 3: Criar o índice que resolve o problema
# CONCURRENTLY permite criar sem bloquear escritas (em produção, sempre usar)
# Aqui usamos sem CONCURRENTLY pois estamos em ambiente de desenvolvimento

CREATE_INDEX = """
CREATE INDEX IF NOT EXISTS idx_propostas_created_at_status
    ON propostas (created_at DESC, status)
"""

run_sql(CREATE_INDEX, fetch=False)
print("Índice criado: idx_propostas_created_at_status")
print()
print("Em produção, use:")
print("  CREATE INDEX CONCURRENTLY IF NOT EXISTS idx_propostas_created_at_status")
print("  ON propostas (created_at DESC, status);")

In [ ]:
# Passo 4: Verificar o novo plano
# O planner deve agora usar o índice criado

print("=== DEPOIS do índice ===")
print()
explain(RELATORIO_LENTO, analyze=True, buffers=True)

In [ ]:
# Passo 5: Comparar os tempos

# Sem índice: forçamos Seq Scan desabilitando temporariamente o índice
with get_connection() as conn:
    with conn.cursor() as cur:
        # Desabilita uso de índice para medir o baseline
        cur.execute("SET enable_indexscan = off; SET enable_bitmapscan = off;")
        cur.execute("SET enable_indexonlyscan = off;")
        start = time.perf_counter()
        cur.execute(RELATORIO_LENTO)
        cur.fetchall()
        time_sem_indice = time.perf_counter() - start

# Com índice
start = time.perf_counter()
run_sql(RELATORIO_LENTO)
time_com_indice = time.perf_counter() - start

print("Comparativo de performance:")
print(f"  Sem índice (Seq Scan) : {time_sem_indice:.3f}s")
print(f"  Com índice            : {time_com_indice:.3f}s")
if time_sem_indice > 0:
    speedup = time_sem_indice / time_com_indice
    print(f"  Melhora               : {speedup:.1f}x mais rápido")

In [ ]:
# Inspecionar os índices existentes em propostas

LISTAR_INDICES = """
SELECT
    indexname,
    indexdef
FROM pg_indexes
WHERE tablename = 'propostas'
ORDER BY indexname
"""

indices = run_sql(LISTAR_INDICES)
print(f"Índices na tabela 'propostas' ({len(indices)} total):")
for idx in indices:
    print(f"  {idx['indexname']}")
    print(f"    {idx['indexdef']}")
    print()

## Resumo

O que olhar primeiro:

`EXPLAIN` (sem ANALYZE) antes de rodar em prod — só mostra o plano, não executa. `EXPLAIN ANALYZE` para medir de verdade.

Se `rows estimado` diverge do real por 10x+, rode `ANALYZE` na tabela para atualizar estatísticas.

`BUFFERS` mostra se a query bate em disco (`shared read`) ou RAM (`shared hit`). Cache frio vs quente pode mudar o tempo em 10x. Hit ratio abaixo de 95% em queries frequentes é sinal de problema.

Seq Scan em tabela grande com WHERE seletivo → provavelmente falta índice. Mas verifique: se a query retorna 40%+ das linhas, índice não vai ajudar (o planner sabe disso e ignora o índice de propósito).

`pg_stat_statements`: ordene por `total_exec_time DESC` para achar o que mais dói no agregado. Por `mean_exec_time DESC` para achar a mais lenta por chamada individual.

`CREATE INDEX CONCURRENTLY` em prod. Sem `CONCURRENTLY` bloqueia escritas na tabela inteira durante a criação do índice.

## Exercícios

Cada exercício apresenta uma query com o plano EXPLAIN real deste banco (2M propostas). Diagnostique e proponha correção.

---

**Exercício 1 — Busca de parcelas em atraso**

```sql
EXPLAIN ANALYZE
SELECT proposta_id, numero, vencimento, valor
FROM parcelas
WHERE vencimento < NOW()
  AND status_pagamento = 'pendente';
```

Plano (com seed corrigido, ~500k parcelas pendentes vencidas):
```
Gather  (cost=1000.00..32250.43 rows=1 width=18) (actual time=61.6..64.2 rows=500000 loops=1)
  Workers Planned: 2
  Workers Launched: 2
  ->  Parallel Seq Scan on parcelas  (cost=0.00..31250.33 rows=1 width=18) (actual time=0.03..55.0 rows=166667 loops=3)
        Filter: (((status_pagamento)::text = 'pendente'::text) AND (vencimento < now()))
        Rows Removed by Filter: 500000
Planning Time: 0.3 ms
Execution Time: 72.1 ms
```

Repare que `rows=1` na estimativa — o planner não sabe quantas parcelas estão em atraso. Que índice criaria?

**Resposta 1**

`status_pagamento` primeiro (igualdade), `vencimento` depois (range):

```sql
CREATE INDEX idx_parcelas_status_vencimento
    ON parcelas (status_pagamento, vencimento);
```

Se invertesse (`vencimento, status_pagamento`), o B-Tree faria o range em `vencimento` primeiro e só depois filtraria `status_pagamento` dentro de cada folha — menos eficiente. Colunas de igualdade sempre antes de colunas de range no índice composto.

Além disso, a estimativa de `rows=1` quando retornaram 500k indica stats ruins — rode `ANALYZE parcelas;` também.

---

**Exercício 2 — Contagem de propostas por cliente**

```sql
EXPLAIN ANALYZE
SELECT cliente_id, COUNT(*)
FROM propostas
WHERE status = 'aprovada'
GROUP BY cliente_id
HAVING COUNT(*) > 3;
```

Plano:
```
HashAggregate  (cost=80657.14..86746.39 rows=57953 width=12) (actual time=217.4..255.7 rows=48492 loops=1)
  Group Key: cliente_id
  Filter: (count(*) > 3)
  Batches: 5  Memory Usage: 8241kB  Disk Usage: 7384kB
  Rows Removed by Filter: 135114
  ->  Seq Scan on propostas  (cost=0.00..52462.00 rows=501247 width=4) (actual time=0.27..140.8 rows=500394 loops=1)
        Filter: ((status)::text = 'aprovada'::text)
        Rows Removed by Filter: 1799606
Planning Time: 0.3 ms
Execution Time: 258.7 ms
```

O Seq Scan é justificado aqui ou um índice ajudaria?

**Resposta 2**

O Seq Scan está parcialmente justificado. `status = 'aprovada'` retorna 500k de 2.3M linhas (~22%). Com essa seletividade, ler via índice (random I/O) pode ser pior que Seq Scan (sequencial). Repare no `Disk Usage: 7384kB` no HashAggregate — o hash não coube em `work_mem` e usou disco.

Um índice `(status, cliente_id)` poderia ajudar via Index Only Scan: o SELECT só precisa de `cliente_id` e `COUNT(*)`, tudo no índice, sem tocar na tabela. Mas com 22% de seletividade, o planner pode ignorar mesmo assim.

Na prática, se essa query roda frequentemente, o melhor investimento é aumentar `work_mem` para evitar o hash spill em disco (os 7384kB de Disk Usage).

---

**Exercício 3 — Estimativa muito errada**

```sql
EXPLAIN ANALYZE
SELECT id, numero, score
FROM propostas
WHERE score BETWEEN 800 AND 1000;
```

Plano (com stats da coluna `score` deletadas):
```
Gather  (cost=1000.00..40237.00 rows=11500 width=24) (actual time=0.19..70.0 rows=700200 loops=1)
  Workers Planned: 2
  Workers Launched: 2
  ->  Parallel Seq Scan on propostas  (cost=0.00..38087.00 rows=4792 width=24) (actual time=0.03..45.9 rows=233400 loops=3)
        Filter: ((score >= 800) AND (score <= 1000))
        Rows Removed by Filter: 533267
Planning Time: 0.3 ms
Execution Time: 82.8 ms
```

O planner estimou 11.500 linhas. Retornaram 700.200. Divergência de 61x. O que explica e como corrigir?

**Resposta 3**

Stats desatualizadas ou ausentes. Sem o histograma de `score`, o planner usa estimativa default para BETWEEN (~0.5% das linhas). Na realidade, score 800-1000 cobre ~30% da tabela.

Correção: `ANALYZE propostas;`

Após o ANALYZE, o plano muda para:
```
Seq Scan on propostas  (cost=0.00..58212.00 rows=704350 width=24) (actual time=0.02..123.0 rows=700200 loops=1)
  Filter: ((score >= 800) AND (score <= 1000))
  Rows Removed by Filter: 1599800
```

Estimativa de 704k vs 700k real — praticamente exata. O Seq Scan continua (30% da tabela não justifica índice), mas agora o planner sabe o que está fazendo e pode dimensionar `work_mem` e joins corretamente.

Em prod, o autovacuum roda ANALYZE automaticamente, mas tabelas com alto volume de INSERTs podem ficar com stats defasadas entre ciclos. Se suspeitar de estimativas ruins, rode `ANALYZE` manualmente.

---

**Exercício 4 — Join entre propostas e decisoes_log**

```sql
EXPLAIN ANALYZE
SELECT p.numero, p.valor, dl.etapa, dl.resultado
FROM propostas p
JOIN decisoes_log dl ON dl.proposta_id = p.id
WHERE p.created_at >= NOW() - INTERVAL '7 days';
```

Plano:
```
Gather  (cost=36299.06..45697.09 rows=4708 width=48) (actual time=205.7..346.8 rows=58356 loops=1)
  Workers Planned: 2
  Workers Launched: 2
  ->  Parallel Hash Join  (cost=35299.06..44226.29 rows=1962 width=48) (actual time=199.9..327.2 rows=19452 loops=3)
        Hash Cond: (dl.proposta_id = p.id)
        ->  Parallel Seq Scan on decisoes_log dl  (cost=0.00..8372.66 rows=211266 width=28) (actual time=0.04..95.9 rows=169013 loops=3)
        ->  Parallel Hash  (cost=35202.33..35202.33 rows=7738 width=28) (actual time=199.1..199.1 rows=6484 loops=3)
              ->  Parallel Seq Scan on propostas p  (cost=0.00..35202.33 rows=7738 width=28) (actual time=0.1..195.4 rows=6484 loops=3)
                    Filter: (created_at >= (now() - '7 days'::interval))
                    Rows Removed by Filter: 660183
Planning Time: 1.9 ms
Execution Time: 350.3 ms
```

Dois Seq Scans: um em propostas (filtro por data), outro em decisoes_log (tabela inteira). Que índice ajudaria e em qual tabela?

**Resposta 4**

O gargalo principal é o Parallel Seq Scan em `propostas` (195ms, varre 660k linhas para achar 6.484). Um índice em `propostas(created_at)` eliminaria esse scan.

Para `decisoes_log`, um índice em `(proposta_id)` permite ao planner trocar o Hash Join por Nested Loop + Index Scan — cada proposta dos últimos 7 dias busca suas 3 decisões direto via índice.

```sql
CREATE INDEX idx_propostas_created_at ON propostas (created_at);
CREATE INDEX idx_decisoes_log_proposta_id ON decisoes_log (proposta_id);
```

Na prática, o planner pode escolher entre Hash Join (melhor se muitas propostas) e Nested Loop (melhor se poucas propostas com índice bom). Com 500k decisoes_log e ~20k propostas no filtro, o Hash Join pode continuar sendo a melhor opção — mas o índice em `propostas(created_at)` elimina o Seq Scan de 195ms em qualquer cenário.

A falta de índice na FK (`decisoes_log.proposta_id`) também prejudica DELETEs em cascata — sempre crie índices nas foreign keys de tabelas com volume.

---

**Exercício 5 — Query de relatório com Sort lento**

```sql
EXPLAIN ANALYZE
SELECT numero, created_at, status, valor
FROM propostas
WHERE status = 'aprovada'
ORDER BY created_at DESC
LIMIT 50;
```

Plano sem índice:
```
Limit  (cost=43691.76..43697.59 rows=50 width=41) (actual time=80.6..83.9 rows=50 loops=1)
  ->  Gather Merge  (cost=43691.76..92867.39 rows=421476 width=41) (actual time=80.6..83.9 rows=50 loops=1)
        Workers Planned: 2
        Workers Launched: 2
        ->  Sort  (cost=42691.73..43218.58 rows=210738 width=41) (actual time=77.7..77.7 rows=40 loops=3)
              Sort Key: created_at DESC
              Sort Method: top-N heapsort  Memory: 31kB
              ->  Parallel Seq Scan on propostas  (cost=0.00..35691.17 rows=210738 width=41) (actual time=0.14..52.6 rows=166798 loops=3)
                    Filter: ((status)::text = 'aprovada'::text)
                    Rows Removed by Filter: 599869
Planning Time: 0.8 ms
Execution Time: 84.0 ms
```

Precisa de apenas 50 linhas, mas lê 500k+ para ordenar e descartar. Qual índice elimina o Seq Scan E o Sort?

**Resposta 5**

```sql
CREATE INDEX idx_propostas_status_created_desc
    ON propostas (status, created_at DESC);
```

`status` primeiro (igualdade), `created_at DESC` depois (ordenação). Com esse índice o plano muda para:

```
Limit  (cost=0.43..11.11 rows=50 width=41) (actual time=0.05..0.28 rows=50 loops=1)
  ->  Index Scan using idx_propostas_status_created_desc on propostas  (cost=0.43..107012.35 rows=500863 width=41) (actual time=0.05..0.27 rows=50 loops=1)
        Index Cond: ((status)::text = 'aprovada'::text)
Planning Time: 1.1 ms
Execution Time: 0.3 ms
```

De 84ms para 0.3ms — **265x mais rápido**. O que aconteceu:

1. O planner vai direto na folha do B-Tree onde `status = 'aprovada'`
2. Lê as entradas já na ordem `created_at DESC` (a direção do índice casa com o ORDER BY)
3. Para após 50 entradas (LIMIT) — não precisa ler as outras 500k

Sem o índice: Seq Scan em 500k linhas + Sort + descarta tudo exceto 50. Com o índice: lê exatamente 50 linhas e para. É o cenário clássico onde índice composto com direção faz diferença dramática: **ORDER BY + LIMIT em tabela grande**.